# Association Rules Mining — Instacart Dataset

This notebook generates and evaluates association rules from the Instacart grocery dataset.
We explore 4 complementary approaches, each targeting a different granularity of co-purchasing behavior:

- Introduction
- I. Data preparation
- II. Baseline - Top N global products
- III. Intra-department rules
- IV. RFM segmentation rules
- V. Cross department rules
- VI. Comparison of approaches
- VII. Limitations and future improvements

## Introduction :

**Key metrics — reminder**

Given a rule A → B :

- **Support** = proportion of all transactions containing both A and B. Measures how frequent the pattern is.

- **Confidence** = proportion of transactions containing A that also contain B. Measures how reliable the rule is.

- **Lift** = confidence / P(B). Measures how much more likely A and B are to co-occur than by random chance. Lift > 1 means the association is stronger than expected. Lift is symmetric: A → B and B → A have the same lift.

**Algorithm:** we use **FP-Growth** to mine frequent itemsets.
FP-Growth was selected over Apriori and ECLAT after preliminary tests:

- **vs. Apriori:** Apriori generates candidate itemsets iteratively and scans the database multiple times. FP-Growth eliminatescandidate generation entirely by compressing the dataset into a FP-Tree, making it significantly faster on large datasets.

- **vs. ECLAT:** ECLAT uses a vertical data format, storing for each productthe list of all transaction IDs where it appears (tidset). Finding frequent itemsets requires intersecting these tidsets. On Instacart, highly frequent products like bananas appear in millions of transactions, making their tidsets enormous and intersections very costly in both memory and time. 
FP-Growth avoids this by sharing common prefixes in the tree — transactions starting with the same products share the same branch, which is well suited to datasets with highly frequent items.

**Parameter tuning**

The parameters `min_support`, `min_confidence`, `min_lift` and `max_transactions` were selected empirically through multiple tests, balancing two constraints:
- generating enough rules to cover a wide range of baskets
- keeping computation time reasonable given the dataset size

Rules are stored with their full metrics (support, confidence, lift) and can therefore be filtered and ranked by any criterion at serving time — in the application we sort by descending lift to surface the strongest associations first, and display only the top K recommendations.

**Note on the objective :** the goal is not to maximize evaluation metrics in isolation but to suggest relevant products that customers might not have thought of, in order to increase basket size. A rule with moderate precision but high lift is more interesting in practice than a high-confidence rule on a product the customer was going to buy anyway — this is why lift is our primary ranking criterion over confidence.

**Memory strategy:** given the size of the dataset (~23M train rows), all transaction preparation and evaluation is done by reading CSV files chunk by chunk — no full DataFrame is kept in memory after the enrichment step.

## I. Data preparation

### I.1. Imports

In [1]:
import pandas as pd
import sys
import os

sys.path.append('../scripts')  # To import from parent directory

from load_data import load_instacart_data
from split_data import temporal_split_instacart
from functions_association_rules import (
    prepare_transactions_from_csv,
    generate_association_rules,
    get_top_products_from_csv,
    evaluate_rules_from_csv,
    print_evaluation_results,
    csv_chunk_generator
)

In [2]:
data = load_instacart_data()

orders = data['orders']
order_products_prior = data['order_products_prior']
order_products_train = data['order_products_train']
products = data['products']
departments = data['departments']

### I.2. Train / test split

We apply a **temporal split** based on `order_number` (70% train / 30% test).
This ensures the model is trained on older orders and evaluated on more recent ones, reflecting the real-world recommendation use case.

With the temporal_split_instacart function we also check that we have the same average basket size and same diversity in baskets between the train and test datasets.

The split is saved to disk and reloaded on subsequent runs to avoid recomputing.

In [3]:
# Check if split already exists to avoid recomputing
if os.path.exists('../data/processed/train.csv') and os.path.exists('../data/processed/test.csv'):
    print("  Split files already exist")
else:
    print("  Creating new split...")
    temporal_split_instacart(
        order_products_prior=order_products_prior,
        order_products_train=order_products_train,
        orders=orders,
        products=products,
        departments=departments,
        train_ratio=0.7,
        save_path='../data/processed/'
    )

  Split files already exist


### I.3. Data enrichment

We merge train and test sets with product and department information to get
`product_name` and `department` columns alongside each transaction row.

These enriched files are saved once and then **read by chunks** throughout the notebook —
the full DataFrames are never kept in memory simultaneously.

In [4]:
import gc

# File paths — used as constants throughout the notebook
TRAIN_ENRICHED_PATH = '../data/processed/train_enriched.csv'
TEST_ENRICHED_PATH  = '../data/processed/test_enriched.csv'
TRAIN_SEG_PATH      = '../data/processed/train_enriched_segments.csv'
TEST_SEG_PATH       = '../data/processed/test_enriched_segments.csv'

# Build enriched files once — merge product/department info into train and test
if os.path.exists(TRAIN_ENRICHED_PATH) and os.path.exists(TEST_ENRICHED_PATH):
    print("  Enriched files already exist")
else:
    print("  Enriching train/test with product and department info...")

    train = pd.read_csv('../data/processed/train.csv')
    train_enriched = train.merge(
        products[['product_id', 'product_name', 'department_id']], on='product_id'
    ).merge(departments[['department_id', 'department']], on='department_id')
    train_enriched.to_csv(TRAIN_ENRICHED_PATH, index=False)
    del train, train_enriched
    gc.collect()

    test = pd.read_csv('../data/processed/test.csv')
    test_enriched = test.merge(
        products[['product_id', 'product_name', 'department_id']], on='product_id'
    ).merge(departments[['department_id', 'department']], on='department_id')
    test_enriched.to_csv(TEST_ENRICHED_PATH, index=False)
    del test, test_enriched
    gc.collect()

    print("  Done")

  Enriched files already exist


## II. Baseline — Top N global products

As a **baseline**, we generate rules on the top 200 most purchased products across all departments,
with no category filtering.

This approach is simple but expected to be biased toward the Produce department,
which represents ~29% of all transactions.

In [26]:
# Prepare transactions on top 200 most purchased products (no department filter)
transactions_general = prepare_transactions_from_csv(
    TRAIN_ENRICHED_PATH,
    top_n_products=200
)

print(f"  Total transactions: {len(transactions_general):,}")

# Generate rules
general_rules = generate_association_rules(
    transactions_general,
    min_support=0.005,
    min_confidence=0.15,
    min_lift=1.3,
    max_transactions=200_000
)

if general_rules is not None:
    general_rules.to_csv('../data/processed/rules_top_products.csv', index=False)
    print(f"  Total rules: {len(general_rules):,}")

    print("\n  Evaluating...")
    metrics_general = evaluate_rules_from_csv(
        rules=general_rules,
        filepath=TEST_ENRICHED_PATH,
        k=10
    )
    print_evaluation_results(metrics_general)
else:
    print("  No rules generated")
    metrics_general = None

    Computing top 200 products (pass 1/2)...
    Building transactions (pass 2/2)...
  Total transactions: 1,831,459
  Total rules: 79

  Evaluating...
    Building test baskets from CSV...
  Precision@10:              9.04%
  Recall@10:                 3.41%
  Coverage:                  50.18%
  Average hits:              0.22
  Baskets evaluated:         10,000
  Baskets with recs:         5,018


As expected, rules are dominated by Produce products (bananas, organic fruits...).
This confirms the need for a more granular approach.

## III. Intra-department rules

We generate rules **within each department separately**, focusing on the top N products
of that department. This avoids cross-department noise and produces more actionable rules
(e.g. 'customers who buy whole milk also buy Greek yogurt' within Dairy).

**Tiered configuration:** departments are grouped by transaction volume.
High-volume departments (Tier 1) get more products and more transactions;
low-volume departments (Tier 3) use lower thresholds to generate enough rules.

**Support thresholds** are set higher for Tier 3 (0.005) than Tier 1-2 (0.003), which may seem counterintuitive. The reasoning is that low-volume departments have fewer transactions, making it easier for spurious co-occurrences to reach a low support threshold by chance. A stricter threshold ensures that rules found in small departments are genuinely robust and not statistical noise.

| Tier | Departments | Top N products | Min support |
|------|-------------|---------------|-------------|
| 1 | produce, dairy eggs, snacks, beverages, frozen | 100 | 0.003 |
| 2 | pantry, household, personal care, bakery, dry goods pasta | 80 | 0.003 |
| 3 | deli, meat seafood, canned goods, international, breakfast, alcohol, babies, pets | 60 | 0.005 |

In [27]:
# Tiered configuration by department volume
TIER_1 = ['produce', 'dairy eggs', 'snacks', 'beverages', 'frozen']
TIER_2 = ['pantry', 'household', 'personal care', 'bakery', 'dry goods pasta']
TIER_3 = ['deli', 'meat seafood', 'canned goods', 'international', 'breakfast', 'alcohol', 'babies', 'pets']

dept_config = {
    **{d: {'n_products': 100, 'max_trans': 150_000, 'supp': 0.003, 'conf': 0.15, 'lift': 1.3} for d in TIER_1},
    **{d: {'n_products': 80,  'max_trans': 100_000, 'supp': 0.003, 'conf': 0.10, 'lift': 1.3} for d in TIER_2},
    **{d: {'n_products': 60,  'max_trans': 50_000,  'supp': 0.005, 'conf': 0.12, 'lift': 1.3} for d in TIER_3}
}

all_dept_rules = []

for i, (dept, config) in enumerate(dept_config.items(), 1):
    print(f"  [{i}/{len(dept_config)}] {dept}...", end=' ')

    # Prepare transactions for this department only (reads CSV by chunks)
    transactions = prepare_transactions_from_csv(
        TRAIN_ENRICHED_PATH,
        filter_column='department',
        filter_values=dept,
        top_n_products=config['n_products']
    )

    rules = generate_association_rules(
        transactions,
        min_support=config['supp'],
        min_confidence=config['conf'],
        min_lift=config['lift'],
        max_transactions=config['max_trans']
    )

    if rules is not None:
        rules['department'] = dept
        all_dept_rules.append(rules)
        print(f"{len(rules)} rules")
    else:
        print("No rules")

# Consolidate all department rules into a single file
dept_rules = pd.concat(all_dept_rules, ignore_index=True)
dept_rules.to_csv('../data/processed/rules_by_department.csv', index=False)
print(f"\n  Total rules by department: {len(dept_rules):,}")

print("\n  Evaluating...")
metrics_dept = evaluate_rules_from_csv(
    rules=dept_rules,
    filepath=TEST_ENRICHED_PATH,
    groupby_column='department',
    k=10
)
print_evaluation_results(metrics_dept)

  [1/18] produce...     Computing top 100 products (pass 1/2)...
    Building transactions (pass 2/2)...
184 rules
  [2/18] dairy eggs...     Computing top 100 products (pass 1/2)...
    Building transactions (pass 2/2)...
20 rules
  [3/18] snacks...     Computing top 100 products (pass 1/2)...
    Building transactions (pass 2/2)...
4 rules
  [4/18] beverages...     Computing top 100 products (pass 1/2)...
    Building transactions (pass 2/2)...
17 rules
  [5/18] frozen...     Computing top 100 products (pass 1/2)...
    Building transactions (pass 2/2)...
6 rules
  [6/18] pantry...     Computing top 80 products (pass 1/2)...
    Building transactions (pass 2/2)...
No rules
  [7/18] household...     Computing top 80 products (pass 1/2)...
    Building transactions (pass 2/2)...
2 rules
  [8/18] personal care...     Computing top 80 products (pass 1/2)...
    Building transactions (pass 2/2)...
4 rules
  [9/18] bakery...     Computing top 80 products (pass 1/2)...
    Building transact

## IV. RFM segmentation rules

We generate rules **per customer segment** (defined by prior RFM analysis). The hypothesis is that purchasing patterns differ between segments — high-frequency customers may have different product affinities than occasional buyers.

**Data preparation:** since the enriched files do not contain the segment column, we add it by mapping `order_id -> user_id -> segment` chunk by chunk, producing dedicated segment-enriched files.

### IV.1 Rules generation

In [28]:
# Load RFM segments (lightweight — only user_id and segment columns)
segments = pd.read_csv('../data/processed/rfm_customer_segments.csv', usecols=['user_id', 'segment'])

In [ ]:
# Add segment column to enriched files — saved once, then reused via chunks
if os.path.exists(TRAIN_SEG_PATH) and os.path.exists(TEST_SEG_PATH):
    print("  Segment files already exist")
else:
    print("  Adding segment column to enriched files...")

    # Build order_id -> segment mapping
    # orders table is small enough to keep in memory
    order_segment = orders[['order_id', 'user_id']].merge(segments, on='user_id', how='left')
    order_segment = order_segment.set_index('order_id')['segment'].to_dict()

    # Add segment column chunk by chunk to both train and test enriched files
    for src_path, dst_path in [(TRAIN_ENRICHED_PATH, TRAIN_SEG_PATH),
                                (TEST_ENRICHED_PATH,  TEST_SEG_PATH)]:
        first_chunk = True
        for chunk in pd.read_csv(src_path, chunksize=100_000):
            chunk['segment'] = chunk['order_id'].map(order_segment)
            chunk.to_csv(dst_path, mode='w' if first_chunk else 'a',
                         index=False, header=first_chunk)
            first_chunk = False

    del order_segment
    gc.collect()
    print("  Done")

In [30]:
# Get segment list without loading the full file
seg_list = pd.read_csv(TRAIN_SEG_PATH, usecols=['segment'])['segment'].dropna().unique()
print(f"  Segments: {list(seg_list)}")

all_seg_rules = []

for i, segment in enumerate(seg_list, 1):
    print(f"  [{i}/{len(seg_list)}] {segment}...", end=' ')

    # Prepare transactions for this segment only (reads CSV by chunks)
    transactions = prepare_transactions_from_csv(
        TRAIN_SEG_PATH,
        filter_column='segment',
        filter_values=segment,
        top_n_products=80
    )

    rules = generate_association_rules(
        transactions,
        min_support=0.005,
        min_confidence=0.15,
        min_lift=1.3,
        max_transactions=100_000
    )

    if rules is not None:
        rules['segment'] = segment
        all_seg_rules.append(rules)
        print(f"{len(rules)} rules")
    else:
        print("No rules")

# Consolidate
if all_seg_rules:
    segment_rules = pd.concat(all_seg_rules, ignore_index=True)
    segment_rules.to_csv('../data/processed/rules_by_segment.csv', index=False)
    print(f"\n  Total rules by segment: {len(segment_rules):,}")

    print("\n  Evaluating...")
    metrics_seg = evaluate_rules_from_csv(
        rules=segment_rules,
        filepath=TEST_SEG_PATH,
        groupby_column='segment',
        k=10
    )
    print_evaluation_results(metrics_seg)
else:
    print("  No segment rules generated")
    metrics_seg = None

  Segments: ['Sleeping', 'Premium', 'New', 'Loyal', 'Frugal', 'Lost', 'Promising', 'High_Check']
  [1/8] Sleeping...     Computing top 80 products (pass 1/2)...
    Building transactions (pass 2/2)...
91 rules
  [2/8] Premium...     Computing top 80 products (pass 1/2)...
    Building transactions (pass 2/2)...
130 rules
  [3/8] New...     Computing top 80 products (pass 1/2)...
    Building transactions (pass 2/2)...
20 rules
  [4/8] Loyal...     Computing top 80 products (pass 1/2)...
    Building transactions (pass 2/2)...
79 rules
  [5/8] Frugal...     Computing top 80 products (pass 1/2)...
    Building transactions (pass 2/2)...
No rules
  [6/8] Lost...     Computing top 80 products (pass 1/2)...
    Building transactions (pass 2/2)...
26 rules
  [7/8] Promising...     Computing top 80 products (pass 1/2)...
    Building transactions (pass 2/2)...
14 rules
  [8/8] High_Check...     Computing top 80 products (pass 1/2)...
    Building transactions (pass 2/2)...
499 rules

  Total 

### IV.2. Unique or shared rules between segments ?

We analyse how many rules are shared across segments vs. unique to a single segment. A high proportion of shared rules would suggest that segmentation adds little value over a global approach.

In [31]:
import matplotlib.pyplot as plt

print("Loading rules by segment...")
segment_rules = pd.read_csv('../data/processed/rules_by_segment.csv')
print(f"  Total rules: {len(segment_rules):,}  |  Segments: {segment_rules['segment'].nunique()}")

# Create unique rule identifier (antecedent -> consequent)
segment_rules['rule_id'] = segment_rules['antecedent'] + ' -> ' + segment_rules['consequent']

# Count how many segments share each rule
rule_counts = segment_rules.groupby('rule_id')['segment'].apply(list).reset_index()
rule_counts['n_segments']    = rule_counts['segment'].apply(len)
rule_counts['segments_list'] = rule_counts['segment'].apply(lambda x: ', '.join(sorted(x)))

# Sharing distribution
print("\nRule sharing distribution:")
sharing_dist = rule_counts['n_segments'].value_counts().sort_index()
for n_seg, count in sharing_dist.items():
    pct = count / len(rule_counts) * 100
    print(f"  Rules in {n_seg} segment(s): {count} ({pct: .1f}%)")

# Unique rules per segment
print("\nUnique rules per segment:")
for seg in seg_list:
    seg_set   = set(segment_rules[segment_rules['segment'] == seg]['rule_id'])
    other_set = set(segment_rules[segment_rules['segment'] != seg]['rule_id'])
    n_total, n_unique = len(seg_set), len(seg_set - other_set)
    if n_total > 0:
        print(f"  {seg}: {n_total} rules total, {n_unique} unique ({n_unique/n_total*100:.1f}%)")


Loading rules by segment...
  Total rules: 859  |  Segments: 7

Rule sharing distribution:
  Rules in 1 segment(s): 414 ( 75.5%)
  Rules in 2 segment(s): 55 ( 10.0%)
  Rules in 3 segment(s): 30 ( 5.5%)
  Rules in 4 segment(s): 24 ( 4.4%)
  Rules in 5 segment(s): 10 ( 1.8%)
  Rules in 6 segment(s): 6 ( 1.1%)
  Rules in 7 segment(s): 9 ( 1.6%)

Unique rules per segment:
  Sleeping: 91 rules total, 5 unique (5.5%)
  Premium: 130 rules total, 17 unique (13.1%)
  New: 20 rules total, 0 unique (0.0%)
  Loyal: 79 rules total, 4 unique (5.1%)
  Lost: 26 rules total, 1 unique (3.8%)
  Promising: 14 rules total, 0 unique (0.0%)
  High_Check: 499 rules total, 387 unique (77.6%)


**Key insights:**

- **75% of rules are unique to a single segment**, which validates the segmentation approach — customer groups do have meaningfully different purchasing patterns, and a global rule set would miss most of them.

- **High_Check stands out drastically:** 499 rules (57% of all rules) with 77.2% unique to this segment. This segment has highly specific co-purchasing behavior that no other group shares. It makes sense because there are more items in their basket so there are more combinatory possibilities.

- **Promising has 0 unique rules**, meaning everything it buys is also bought by other segments. This segment may not benefit from segment-specific recommendations — global or department-based rules would be sufficient.

- **Sleeping, Loyal and Lost have very few unique rules (< 4%)**, suggesting that their purchasing patterns are close to the average customer. The value of segmentation for these groups is limited.

**Implication:** segmentation is most valuable for High_Check customers.
For the other segments, the gain over a department-based approach is marginal in terms of rule specificity.

## V. Cross-department rules

The previous approaches generate rules **within** a single department or segment.
Here we look for associations **between** departments — e.g. snacks & beverages, dairy & bakery, personal care & babies.

**Methodology:**
For each department pair `(A, B)`:
1. Identify the top 100 products of each department separately
2. Keep only baskets containing at least one product from **each** department
3. Generate rules on these mixed baskets
4. Retain only rules where antecedent and consequent come from **different departments** and where each side comes from a single department (no mixed-department antecedents)

**Note on support:** values are computed on the filtered subset of mixed baskets, not on the full dataset — they are therefore higher than they would be globally.

**Chosen pairs** are based on intuitive complementarity and basesd on some department association found during previous try (breakfast staples, snacking occasions, baby care, etc.).

In [5]:
# Department pairs to study
DEPT_PAIRS = [
    ('snacks',        'beverages'),
    ('produce',       'meat seafood'),
    ('snacks',        'alcohol'),
    ('personal care', 'babies'),
    ('dairy eggs',    'bakery'),
    ('frozen',        'beverages'),
    ('produce',       'dairy eggs'),
    ('breakfast',     'dairy eggs'),
]

# Shared configuration for all pairs
PAIR_CONFIG = {
    'n_products': 100,   # top N products per department
    'max_trans':  150_000,
    'supp':       0.003,
    'conf':       0.10,
    'lift':       1.3
}

all_pair_rules = []

for i, (dept_a, dept_b) in enumerate(DEPT_PAIRS, 1):
    print(f"  [{i}/{len(DEPT_PAIRS)}] {dept_a} x {dept_b}...", end=' ')

    # Get top N products for each department separately
    top_a = get_top_products_from_csv(
        TRAIN_ENRICHED_PATH, top_n=PAIR_CONFIG['n_products'],
        filter_column='department', filter_values=dept_a
    )
    top_b = get_top_products_from_csv(
        TRAIN_ENRICHED_PATH, top_n=PAIR_CONFIG['n_products'],
        filter_column='department', filter_values=dept_b
    )
    allowed_products = top_a | top_b

    # Build mixed transactions: keep only baskets with products from BOTH departments
    transactions_dict = {}
    dept_presence = {}   # order_id -> set of departments present

    for chunk in csv_chunk_generator(TRAIN_ENRICHED_PATH):
        chunk_filtered = chunk[chunk['product_name'].isin(allowed_products)]
        for order_id, group in chunk_filtered.groupby('order_id'):
            if order_id not in transactions_dict:
                transactions_dict[order_id] = []
                dept_presence[order_id] = set()
            transactions_dict[order_id].extend(group['product_name'].tolist())
            dept_presence[order_id].update(group['department'].tolist())

    # Only keep baskets with products from BOTH departments
    transactions = [
        prods for oid, prods in transactions_dict.items()
        if dept_a in dept_presence[oid] and dept_b in dept_presence[oid]
    ]

    print(f"{len(transactions):,} mixed baskets...", end=' ')

    if len(transactions) < 100:
        print("Not enough baskets, skipping")
        continue

    rules = generate_association_rules(
        transactions,
        min_support=PAIR_CONFIG['supp'],
        min_confidence=PAIR_CONFIG['conf'],
        min_lift=PAIR_CONFIG['lift'],
        max_transactions=PAIR_CONFIG['max_trans']
    )

    if rules is not None:
        # Filter to keep only truly cross-department rules
        # Using top_a and top_b directly avoids product_dept_map and the
        # comma-in-product-name bug (e.g. "Peach, Apricot & Banana Baby Food")
        def all_from_a(product_str):
            return all(p in top_a for p in product_str.split(', '))

        def all_from_b(product_str):
            return all(p in top_b for p in product_str.split(', '))

        inter_rules = rules[
            ((rules['antecedent'].apply(all_from_a)) & (rules['consequent'].apply(all_from_b))) |
            ((rules['antecedent'].apply(all_from_b)) & (rules['consequent'].apply(all_from_a)))
        ].copy()

        # Assign department based on known sets — no string parsing needed
        inter_rules['antecedent_dept'] = inter_rules['antecedent'].apply(
            lambda x: dept_a if all_from_a(x) else dept_b)
        inter_rules['consequent_dept'] = inter_rules['consequent'].apply(
            lambda x: dept_b if all_from_b(x) else dept_a)

        inter_rules['pair'] = f"{dept_a} x {dept_b}"
        all_pair_rules.append(inter_rules)
        print(f"{len(inter_rules)} inter-dept rules")
    else:
        print("No rules")


# Consolidate
if all_pair_rules:
    pair_rules = pd.concat(all_pair_rules, ignore_index=True)
    pair_rules.to_csv('../data/processed/rules_cross_department_pairs.csv', index=False)
    print(f"\n  Total cross-department rules: {len(pair_rules):,}")

else:
    print("  No cross-department rules generated")
    pair_rules = None

  [1/8] snacks x beverages...     Computing top 100 products...
    Computing top 100 products...
166,208 mixed baskets... 25 inter-dept rules
  [2/8] produce x meat seafood...     Computing top 100 products...
    Computing top 100 products...
287,018 mixed baskets... 88 inter-dept rules
  [3/8] snacks x alcohol...     Computing top 100 products...
    Computing top 100 products...
7,469 mixed baskets... 39 inter-dept rules
  [4/8] personal care x babies...     Computing top 100 products...
    Computing top 100 products...
3,969 mixed baskets... 26 inter-dept rules
  [5/8] dairy eggs x bakery...     Computing top 100 products...
    Computing top 100 products...
237,233 mixed baskets... 9 inter-dept rules
  [6/8] frozen x beverages...     Computing top 100 products...
    Computing top 100 products...
151,409 mixed baskets... 2 inter-dept rules
  [7/8] produce x dairy eggs...     Computing top 100 products...
    Computing top 100 products...
795,427 mixed baskets... 95 inter-dept ru

Here we don't evaluate on the test dataset because with our method, the results will be much higher as we filtered on 2 departments each time.

## VI. Comparison of approaches

We compare the 3 main approaches (baseline, by department, by segment) on the same evaluation metrics.

**Evaluation methodology (Shani & Gunawardana, 2011):**
Each test basket is split 50/50 — the first half serves as input (known items),
the second half as ground truth (items to predict).
Rules are applied to the known items to generate up to K=10 recommendations.

**Metrics:**
- **Precision@10**: what proportion of the 10 recommendations are actually purchased?
- **Recall@10**: what proportion of actual purchases were recommended?
- **Coverage**: what proportion of baskets receive at least one recommendation?

**Limitation:** metrics are computed on top N products only — baskets with products
outside the top N cannot receive recommendations, which artificially lowers coverage
and recall.

In [34]:
comparison = pd.DataFrame([
    {
        'Approach':     'Top N Global',
        'Rules':        len(general_rules) if general_rules is not None else 0,
        'Precision@10': f"{metrics_general['precision@K']:.2%}" if metrics_general else 'N/A',
        'Recall@10':    f"{metrics_general['recall@K']:.2%}"    if metrics_general else 'N/A',
        'Coverage':     f"{metrics_general['coverage']:.2%}"    if metrics_general else 'N/A',
        'Avg Hits':     f"{metrics_general['avg_hits']:.2f}"    if metrics_general else 'N/A'
    },
    {
        'Approach':     'By Department',
        'Rules':        len(dept_rules),
        'Precision@10': f"{metrics_dept['precision@K']:.2%}",
        'Recall@10':    f"{metrics_dept['recall@K']:.2%}",
        'Coverage':     f"{metrics_dept['coverage']:.2%}",
        'Avg Hits':     f"{metrics_dept['avg_hits']:.2f}"
    },
    {
        'Approach':     'By Segment',
        'Rules':        len(segment_rules) if all_seg_rules else 0,
        'Precision@10': f"{metrics_seg['precision@K']:.2%}" if metrics_seg else 'N/A',
        'Recall@10':    f"{metrics_seg['recall@K']:.2%}"    if metrics_seg else 'N/A',
        'Coverage':     f"{metrics_seg['coverage']:.2%}"    if metrics_seg else 'N/A',
        'Avg Hits':     f"{metrics_seg['avg_hits']:.2f}"    if metrics_seg else 'N/A'
    }
])

print(comparison.to_string(index=False))

     Approach  Rules Precision@10 Recall@10 Coverage Avg Hits
 Top N Global     79        9.04%     3.41%   50.18%     0.22
By Department    474        8.00%     4.09%   26.84%     0.24
   By Segment    859        8.12%     3.50%   53.98%     0.23


### Results analysis

**Precision (~8-9%)** is consistent across all approaches. Rules are not personalized, they apply the same patterns to all customers within a group.

**Recall (~3.5-4%)** is low but not surprising: rules are built on top N products only, so many target items fall outside the scope of any rule.

**Coverage** is the most discriminating metric:
- By Department (27%) is penalized by the `first` department assignment — baskets spanning multiple departments only receive rules from one of them.
- By Segment (54%) and  Top N Global (49%) perform better because their rules apply more broadly across baskets.

**Avg Hits (0.23)** is identical across approaches — on average, rules generate less than 1 correct recommendation per basket. 
This reflects the limitation of non-personalized association rules on a 50k-product catalog.

**Cross-departments** are not included in this comparison — their metrics are upward biased since rules are generated on baskets pre-filtered to contain products from both departments, which artificially inflates support and coverage.

**Key takeaway :** the three approaches are complementary rather than competing :

- Intra-department rules are more precise within a category
- Segment rules improve coverage by tailoring recommendations to customer profiles
- Cross-department rules open bundling opportunities between categories that the other approaches miss entirely

## VII. Limitations and future improvements

### VII.1. Evaluation improvements

The current evaluation has two known limitations worth addressing:
- **Department assignment:** baskets spanning multiple departments only receive rules from the first department encountered. A better approach would apply rules from ALL departments represented in the known items.

### VII.2. Better use of the `reordered` feature

The dataset contains a `reordered` flag that we did not exploit.
It could have been used to distinguish:
- **Reordered = 1:** planned, habitual purchases — less interesting for recommendation since the customer already knows the product
- **Reordered = 0:** discovery purchases — more valuable targets for association rules, as they represent genuine new additions to the basket

Filtering target items to `reordered = 0` during evaluation would give more realistic picture of the rules' ability to drive product discovery.

### VII.3. Richer customer segmentation

RFM captures recency, frequency and monetary value but ignores behavioral and basket diversity dimensions. Potential improvements:
- **Basket diversity score:** customers who shop across many departments vs. those who stick to a few categories have very different recommendation needs
- **Category affinity profiles:** cluster customers based on their department purchase distribution rather than just order frequency
- **Implicit demographic inference:** purchase patterns could reveal customer profiles without explicit demographic data : customers buying baby products (babies, personal care) likely have young children; customers
  buying predominantly organic produce, supplements and personal care may skew toward a health-conscious profile. 

### VII.4. Moving toward a machine learning approach

Association rules are interpretable but not personalized — the same ruleapplies to all customers in a group. A ML recommendation model would allowtrue personalization. Relevant features would include:

- **User-level:** RFM scores, nb of orders, avg basket size, preferred departments, reorder rate, avg days between orders
- **Product-level:** overall popularity, reorder rate, department, avg position in cart (first-added vs. last-added)
- **Interaction-level:** has the user bought this product before, how long ago, how often

Suggested approach: XGBoost with the features above for a ranking model. The `reordered` flag would be a natural binary target for supervised learning.